# Vígil.ia — Fine-tune MULTI-GRÃO com as capturas reais (`soja pra treino`)

Fine-tune do **`soja_yolo11n_base12k_v2.pt`** com as capturas do `vigil_deck.py`
revisadas por você — **com foco em cena multi-grão**, que é o alvo do projeto.

**Como os dados são usados (v2 deste notebook):**
- rótulo = **pasta** (sua correção); nome do arquivo (previsão antiga) é ignorado
- os recortes viram **matéria-prima de cenas sintéticas multi-grão**: cada cena
  sorteia o quanto os grãos se encostam — de solto (~10%) até **encostado** (~45%,
  14-35 grãos por cena) — o alvo é reconhecer um grão NO MEIO de outros
- **caixa apertada (foco desta versão):** a máscara do recorte é limpa e erodida
  (tira o halo de fundo) e a borda é colada crispa na cena → a caixa cola no grão,
  não numa auréola. Fundo escuro c/ gradiente + sombra por grão imitam o vídeo real
- recorte solto só entra como imagem de treino se o Otsu achar uma caixa REAL
  (< 75% do quadro); recorte apertado com "caixa = quadro inteiro" NÃO vira imagem
  de treino (esse era o vício documentado do base_12k) — vira só matéria de cena
- **oversampling** até a classe majoritária (o balanceamento do projeto; ultralytics
  não tem class weights nativo) — vale também pra frequência nas cenas
- **val = cenas multi-grão feitas com recortes do split de val** (grãos que o treino
  nunca viu) → o mAP agora MEDE multi-grão, e o placar A/B contra o campeão é justo

No fim: **placar A/B** + export OpenVINO. GPU: L4 ou A100.

## 0. Setup

In [ ]:
!pip -q install "ultralytics==8.4.80"
import ultralytics
ultralytics.checks()

## 1. Caminhos + config

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')

# campeão atual (ponto de partida do fine-tune) — o mesmo .pt que roda no Deck/Windows
CHAMPION_PT = '/content/drive/MyDrive/soja_yolo11n_base12k_v2.pt'
assert os.path.exists(CHAMPION_PT), f'campeão não encontrado: {CHAMPION_PT}'

# capturas do vigil_deck.py, revisadas por você (rótulo = PASTA)
REAL_SRCS = ['/content/drive/MyDrive/soja pra treino']
for p in REAL_SRCS:
    assert os.path.isdir(p), f'pasta não encontrada: {p}'

SIZE = 640
BATCH = 64          # batch alto; na A100 suba p/ 128 (ou use batch=-1 p/ auto)
N_SYNTH = 1200      # cenas multi-grão de TREINO (o prato principal agora)
N_VAL_SCENES = 120  # cenas multi-grão de VAL (recortes do split val, nunca treinados)
BLUR_FRAC = 0.4
VAL_FRAC = 0.15     # mesmo split por hash do treino anterior (determinístico)
print('partida:', CHAMPION_PT)
print('dados  :', REAL_SRCS)

## 2. Funções de dataset (idênticas ao `treino_campeao_640.ipynb`)

`class_of` entende os nomes PT das pastas (`intacto`, `quebrado`, `manchado`,
`não maduro`, `casca danificada`). O nome do arquivo NÃO é usado como rótulo.

In [ ]:
import glob, hashlib, unicodedata, cv2, yaml
import numpy as np

NAMES = ['broken', 'immature', 'intact', 'skin-damaged', 'spotted']
ALIASES = {0: ['broken', 'quebrad'], 1: ['immature', 'imatur', 'nao maduro'],
           2: ['intact'], 3: ['skin', 'casca', 'ardid', 'danific'], 4: ['spotted', 'manchad']}
IGNORE = ['part of the original']
IMG_EXT = ('.jpg', '.jpeg', '.png', '.bmp', '.webp')
RNG = np.random.default_rng(42)

def norm(s):
    return unicodedata.normalize('NFKD', s).encode('ascii', 'ignore').decode().lower()

def class_of(folder):
    n = norm(folder)
    if any(norm(k) in n for k in IGNORE):
        return None
    for idx in range(5):
        if any(norm(k) in n for k in ALIASES[idx]):
            return idx
    return None

def collect_real(srcs, val_frac=VAL_FRAC):
    items = []
    for src in srcs:
        for root, _, files in os.walk(src):
            cls = None
            for part in reversed(root.split(os.sep)):
                c = class_of(part)
                if c is not None:
                    cls = c; break
            if cls is None:
                continue
            for fn in files:
                if fn.lower().endswith(IMG_EXT):
                    p = os.path.join(root, fn)
                    h = int(hashlib.md5(p.encode()).hexdigest(), 16)
                    items.append((p, cls, 'val' if (h % 100) < val_frac * 100 else 'train'))
    from collections import Counter
    print('split:', dict(Counter(sp for _, _, sp in items)))
    print('por classe:', {NAMES[c]: n for c, n in
                          sorted(Counter(c for _, c, _ in items).items())})
    return items

def sat_box(img):
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    s = cv2.GaussianBlur(hsv[:, :, 1], (5, 5), 0)
    _, th = cv2.threshold(s, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    th = cv2.morphologyEx(th, cv2.MORPH_OPEN, np.ones((5, 5), np.uint8))
    cnts, _ = cv2.findContours(th, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not cnts:
        return None
    c = max(cnts, key=cv2.contourArea)
    area = cv2.contourArea(c)
    h, w = img.shape[:2]
    if area < 0.01 * h * w or area > 0.90 * h * w:
        return None
    x, y, bw, bh = cv2.boundingRect(c)
    pad = int(0.04 * min(bw, bh)) + 2
    x1, y1 = max(0, x - pad), max(0, y - pad)
    x2, y2 = min(w, x + bw + pad), min(h, y + bh + pad)
    return (((x1 + x2) / 2) / w, ((y1 + y2) / 2) / h, (x2 - x1) / w, (y2 - y1) / h)

def otsu_box(img):
    h, w = img.shape[:2]
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    blur = cv2.GaussianBlur(gray, (5, 5), 0)
    _, th = cv2.threshold(blur, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    cnts, _ = cv2.findContours(th, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not cnts:
        return None
    c = max(cnts, key=cv2.contourArea)
    area = cv2.contourArea(c)
    if area < 0.005 * h * w or area > 0.995 * h * w:
        return None
    x, y, bw, bh = cv2.boundingRect(c)
    pad = int(0.04 * min(bw, bh)) + 2
    x1, y1 = max(0, x - pad), max(0, y - pad)
    x2, y2 = min(w, x + bw + pad), min(h, y + bh + pad)
    return (((x1 + x2) / 2) / w, ((y1 + y2) / 2) / h, (x2 - x1) / w, (y2 - y1) / h)

def full_box(img):
    # fallback p/ recorte apertado do vigil_deck (grão ocupa o quadro todo e o
    # Otsu recusa por area > 0.90): a caixa É o quadro, com folga mínima
    return (0.5, 0.5, 0.96, 0.96)

def letterbox640(img, size=640):
    h, w = img.shape[:2]
    s = size / max(h, w)
    img = cv2.resize(img, (max(1, round(w * s)), max(1, round(h * s))))
    h, w = img.shape[:2]
    top, left = (size - h) // 2, (size - w) // 2
    img = cv2.copyMakeBorder(img, top, size - h - top, left, size - w - left,
                             cv2.BORDER_CONSTANT, value=(0, 0, 0))
    return img, s, left, top

def motion_blur(img, rng=RNG):
    k = int(rng.choice([7, 9, 11, 13, 15]))
    kernel = np.zeros((k, k), np.float32)
    kernel[k // 2, :] = 1.0
    M = cv2.getRotationMatrix2D((k / 2 - 0.5, k / 2 - 0.5), float(rng.uniform(0, 180)), 1)
    kernel = cv2.warpAffine(kernel, M, (k, k))
    kernel /= max(kernel.sum(), 1e-6)
    return cv2.filter2D(img, -1, kernel)

def extract_cutout(img):
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    s = cv2.GaussianBlur(hsv[:, :, 1], (5, 5), 0)
    _, th = cv2.threshold(s, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    th = cv2.morphologyEx(th, cv2.MORPH_OPEN, np.ones((5, 5), np.uint8))
    cnts, _ = cv2.findContours(th, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not cnts:
        return None
    c = max(cnts, key=cv2.contourArea)
    area = cv2.contourArea(c)
    h, w = img.shape[:2]
    if area < 0.01 * h * w or area > 0.90 * h * w:
        return None
    bx, by, bw, bh = cv2.boundingRect(c)
    # solidez: grão é compacto; blob esfarrapado (área << caixa) daria caixa larga -> rejeita
    if area < 0.55 * bw * bh:
        return None
    mask = np.zeros((h, w), np.uint8)
    cv2.drawContours(mask, [c], -1, 255, -1)
    # erode 1px: puxa a borda pra dentro e tira o halo -> a caixa cola no grão
    mask = cv2.erode(mask, np.ones((3, 3), np.uint8))
    ys, xs = np.where(mask > 0)
    if not len(xs):
        return None
    y0, y1 = ys.min(), ys.max() + 1
    x0, x1 = xs.min(), xs.max() + 1
    # recorta na bbox APERTADA da máscara já limpa
    return img[y0:y1, x0:x1], mask[y0:y1, x0:x1]


def make_bg(size, rng):
    # fundo ESCURO como o vídeo real (fundo preto/cinza), com gradiente de luz de cima + textura
    base = int(rng.integers(8, 70))
    canvas = np.full((size, size, 3), float(base), np.float32)
    grad = np.linspace(rng.uniform(-15, 5), rng.uniform(-5, 15), size)[:, None, None]
    canvas += grad
    canvas += rng.normal(0, 8, (size, size, 3))
    return np.clip(canvas, 0, 255).astype(np.uint8)


def make_scene(cutouts, rng=RNG, size=640, overlap_max=None):
    # overlap_max varia por cena: de "solto" a "encostado". Teto 0.45 (não 0.55):
    # oclusão extrema ensina caixa alucinada; aqui a prioridade é caixa que cola no grão
    if overlap_max is None:
        overlap_max = float(rng.uniform(0.10, 0.45))
    canvas = make_bg(size, rng)
    occ = np.zeros((size, size), np.uint8)
    boxes = []
    n_graos = int(rng.integers(6, 26)) if overlap_max < 0.30 else int(rng.integers(14, 36))
    for _ in range(n_graos):
        cls, crop, mask = cutouts[int(rng.integers(len(cutouts)))]
        s = int(rng.integers(60, 150)) / max(crop.shape[:2])
        crop2 = cv2.resize(crop, None, fx=s, fy=s)
        mask2 = cv2.resize(mask, None, fx=s, fy=s, interpolation=cv2.INTER_NEAREST)
        h2, w2 = crop2.shape[:2]
        diag = int(np.ceil(np.hypot(h2, w2))) + 2
        M = cv2.getRotationMatrix2D((w2 / 2, h2 / 2), float(rng.uniform(0, 360)), 1)
        M[0, 2] += (diag - w2) / 2
        M[1, 2] += (diag - h2) / 2
        crop3 = cv2.warpAffine(crop2, M, (diag, diag))
        mask3 = cv2.warpAffine(mask2, M, (diag, diag), flags=cv2.INTER_NEAREST)
        ys, xs = np.where(mask3 > 0)
        if not len(xs):
            continue
        crop3 = crop3[ys.min():ys.max() + 1, xs.min():xs.max() + 1]
        mask3 = mask3[ys.min():ys.max() + 1, xs.min():xs.max() + 1]
        gh, gw = mask3.shape
        if gh >= size - 2 or gw >= size - 2:
            continue
        placed = False
        for _try in range(25):
            px = int(rng.integers(0, size - gw))
            py = int(rng.integers(0, size - gh))
            inter = (occ[py:py + gh, px:px + gw] > 0) & (mask3 > 0)
            if inter.sum() <= overlap_max * (mask3 > 0).sum():
                placed = True
                break
        if not placed:
            continue
        # sombreamento leve por grão: não fica com cara de "colado", ajuda a separar do fundo
        shade = float(rng.uniform(0.75, 1.1))
        crop3 = np.clip(crop3.astype(np.float32) * shade, 0, 255).astype(np.uint8)
        # feather MENOR (3,3): borda crispa -> o grão visível casa com a caixa, sem halo
        alpha = (cv2.GaussianBlur(mask3, (3, 3), 0).astype(np.float32) / 255)[..., None]
        reg = canvas[py:py + gh, px:px + gw]
        canvas[py:py + gh, px:px + gw] = (alpha * crop3 + (1 - alpha) * reg).astype(np.uint8)
        occ[py:py + gh, px:px + gw][mask3 > 0] = 255
        boxes.append((cls, (px + gw / 2) / size, (py + gh / 2) / size, gw / size, gh / size))
    return canvas, boxes


def balance_train(items):
    from collections import defaultdict
    train = [it for it in items if it[2] == 'train']
    rest = [it for it in items if it[2] != 'train']
    by = defaultdict(list)
    for it in train:
        by[it[1]].append(it)
    mx = max(len(v) for v in by.values())
    out = []
    for c, v in by.items():
        out += v + [v[int(i)] for i in RNG.integers(0, len(v), mx - len(v))]
    print('oversampling (train):', {NAMES[c]: sum(1 for it in out if it[1] == c)
                                    for c in sorted(by)})
    return out + rest

## 3. Constrói o dataset multi-grão

Treino: cenas sintéticas (recortes do split train) + os poucos recortes com caixa real.
Val: cenas sintéticas feitas SÓ com recortes do split val (grão nunca visto no treino).
Ressalva honesta: val sintético mede a cena do compositor, não o mundo — o juiz final
continua sendo o vídeo real no `vigil_deck.py`.

In [ ]:
def build_ft(items, out_dir, n_synth=N_SYNTH, n_val_scenes=N_VAL_SCENES,
             blur_frac=BLUR_FRAC):
    assert items, 'Nenhuma imagem coletada! Confira REAL_SRCS.'
    import shutil
    shutil.rmtree(out_dir, ignore_errors=True)
    for sp in ('train', 'val'):
        os.makedirs(f'{out_dir}/images/{sp}', exist_ok=True)
        os.makedirs(f'{out_dir}/labels/{sp}', exist_ok=True)
    items = balance_train(items)

    cut_train, cut_val = [], []
    singles = skipped = 0
    for i, (path, cls, sp) in enumerate(items):
        if i % 200 == 0:
            print(f'  fotos {i}/{len(items)}…', flush=True)
        img = cv2.imread(path)
        if img is None:
            skipped += 1; continue
        cut = extract_cutout(img)
        if cut is not None:
            (cut_train if sp == 'train' else cut_val).append((cls, cut[0], cut[1]))
        # recorte solto só vira IMAGEM de treino se tiver caixa real (não o quadro
        # inteiro) — senão ensina o vício "caixa = quadro" que já derrubou o base_12k
        if sp != 'train':
            continue
        box = sat_box(img) or otsu_box(img)
        if box is None or box[2] * box[3] > 0.75:
            continue
        h0, w0 = img.shape[:2]
        lb, s, left, top = letterbox640(img)
        cx, cy, ww, hh = box
        cx = (cx * w0 * s + left) / 640.0
        cy = (cy * h0 * s + top) / 640.0
        ww = (ww * w0 * s) / 640.0
        hh = (hh * h0 * s) / 640.0
        line = f'{cls} {cx:.6f} {cy:.6f} {ww:.6f} {hh:.6f}'
        stem = f'single_{i:06d}'
        cv2.imwrite(f'{out_dir}/images/train/{stem}.jpg', lb, [cv2.IMWRITE_JPEG_QUALITY, 95])
        open(f'{out_dir}/labels/train/{stem}.txt', 'w').write(line)
        cv2.imwrite(f'{out_dir}/images/train/{stem}b.jpg', motion_blur(lb),
                    [cv2.IMWRITE_JPEG_QUALITY, 95])
        open(f'{out_dir}/labels/train/{stem}b.txt', 'w').write(line)
        singles += 2
    print(f'singles c/ caixa real: {singles} | skipped: {skipped} | '
          f'recortes train: {len(cut_train)} | recortes val: {len(cut_val)}')
    assert cut_train, 'Nenhum recorte de treino extraído!'
    assert cut_val, ('Nenhum recorte de VAL extraído — sem como montar o val '
                     'multi-grão. Confira as imagens do split val.')

    def write_scenes(cutouts, split, n, rng, blur):
        made = 0
        for j in range(n):
            if j % 200 == 0:
                print(f'  cenas {split} {j}/{n}…', flush=True)
            canvas, boxes = make_scene(cutouts, rng=rng)
            if not boxes:
                continue
            if rng.random() < blur:
                canvas = motion_blur(canvas, rng=rng)
            stem = f'synth_{split}_{j:05d}'
            cv2.imwrite(f'{out_dir}/images/{split}/{stem}.jpg', canvas,
                        [cv2.IMWRITE_JPEG_QUALITY, 95])
            open(f'{out_dir}/labels/{split}/{stem}.txt', 'w').write(
                '\n'.join(f'{c} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}'
                           for c, cx, cy, w, h in boxes))
            made += 1
        print(f'cenas {split}: {made}')

    write_scenes(cut_train, 'train', n_synth, np.random.default_rng(42), blur_frac)
    write_scenes(cut_val, 'val', n_val_scenes, np.random.default_rng(123), blur_frac)

OUT = '/content/soja_ft_capturas'
items = collect_real(REAL_SRCS)
build_ft(items, OUT)

FT_YAML = f'{OUT}/data.yaml'
yaml.safe_dump({'train': f'{OUT}/images/train', 'val': f'{OUT}/images/val',
                'names': {i: n for i, n in enumerate(NAMES)}},
               open(FT_YAML, 'w'), sort_keys=False, allow_unicode=True)
print('data.yaml:', FT_YAML)

## 4. Fine-tune (mesma receita/augmentation do campeão, batch alto)

In [ ]:
import shutil
from ultralytics import YOLO

COMMON = dict(
    imgsz=SIZE, device=0, seed=42, optimizer='AdamW',
    cache=False, workers=8,
    mosaic=1.0, hsv_v=0.5, degrees=15, translate=0.1, scale=0.5,
    fliplr=0.5, flipud=0.5,
    project='runs_ft_capturas', exist_ok=True,
)

DST = '/content/drive/MyDrive/soja_yolo11n_multi_v3.pt'
if os.path.exists(DST):
    print('já treinado:', DST)
else:
    m = YOLO(CHAMPION_PT)
    m.train(name='11n_ft_capturas', data=FT_YAML, batch=BATCH, epochs=60,
            lr0=0.0005,          # metade do lr do campeão: é fine-tune de modelo já bom
            patience=20, close_mosaic=8, **COMMON)
    shutil.copy(str(m.trainer.best), DST)
    print('salvo no Drive:', DST)

## 5. Placar A/B — campeão vs fine-tunado (val MULTI-GRÃO)

Agora o val são cenas com vários grãos (recortes que o treino nunca viu) — mede
exatamente o alvo do projeto, e o campeão pontua de verdade (comparação justa).

In [ ]:
from ultralytics import YOLO

def score(pt, label):
    r = YOLO(pt).val(data=FT_YAML, imgsz=SIZE, verbose=False)
    print(f'{label:28s} mAP50={r.box.map50:.3f}  mAP50-95={r.box.map:.3f}')
    # ap50 vem alinhado a ap_class_index (só classes presentes no val)
    for j, ci in enumerate(r.box.ap_class_index):
        print(f'   {NAMES[int(ci)]:14s} mAP50={r.box.ap50[j]:.3f}')
    return r.box.map50

print('=== PLACAR (val = suas capturas) ===')
a = score(CHAMPION_PT, 'base12k_v2 (campeão atual)')
b = score(DST,         'base12k_v3 (fine-tunado)')
print()
print('✅ v3 venceu — adote' if b > a else '⚠️ v3 NÃO superou — mantenha o v2 e colete mais dado')

## 6. Export OpenVINO (p/ a iGPU Intel no Windows)

Baixa a pasta `soja_yolo11n_base12k_v3_openvino_model` e usa no `vigil_deck.py`
com `--model soja_yolo11n_base12k_v3_openvino_model --device intel:gpu`.

In [ ]:
!yolo export model={DST} format=openvino imgsz=640 half=True
import shutil
src = DST.replace('.pt', '_openvino_model')
shutil.make_archive(src, 'zip', src)
print('zip no Drive:', src + '.zip')

## 5b. Juiz de verdade — `teste_soja.mp4` (vídeo real, campeão vs novo)

Diferente do app ao vivo (que trava a classe enquanto você segura a câmera parada
por alguns segundos), aqui o **vídeo inteiro já existe** — então fazemos **2 passadas**:

1. **Passada 1 (votos):** roda o tracking no vídeo todo e junta os votos de cada grão.
2. **Passada 2 (render):** desenha cada grão **já com o veredito final desde o 1º frame**
   — sem "analisando...", sem esperar 3s. As caixas são suavizadas (EMA) pra não tremer,
   e grãos vistos por poucos frames (ruído) não são desenhados.

Mesma regra exigente do `vigil_deck.py` (defeito precisa provar; intacto = benefício
da dúvida). Gera 2 vídeos no Drive pra assistir lado a lado.

> Honestidade: avaliação **visual**, não benchmark rotulado — não há gabarito por grão
> nesse vídeo. Mede alucinação e estabilidade de caixa a olho, o critério final do projeto.


In [ ]:
import os
from collections import defaultdict, Counter
import cv2
from ultralytics import YOLO

def first_existing(*paths):
    return next((p for p in paths if os.path.exists(p)), None)

VIDEO_TESTE = first_existing('/content/drive/MyDrive/teste_soja.mp4',
                             '/content/drive/MyDrive/teste_soja.avi')
assert VIDEO_TESTE, 'teste_soja.(mp4|avi) não encontrado no Drive!'

# regra exigente por classe (igual ao vigil_deck.py)
RATIOS = {'broken': 0.85, 'skin-damaged': 0.80, 'spotted': 0.75, 'immature': 0.75}
MIN_TRACK_FRAMES = 3    # grão visto em menos frames que isso = ruído (não desenha)
SMOOTH = 0.4            # suavização da caixa (EMA): menor = mais estável, menos treme
CONF = 0.30             # baixe p/ 0.25 se faltar caixa; suba p/ 0.4 se aparecer lixo
PT_LABEL = {'broken': 'Quebrado', 'immature': 'Imaturo', 'intact': 'Intacto',
            'skin-damaged': 'Casca danif.', 'spotted': 'Manchado'}
COLORS = {'intact': (90, 200, 90), 'immature': (60, 200, 200), 'broken': (170, 100, 210),
          'skin-damaged': (255, 160, 60), 'spotted': (70, 70, 235)}

def veredito(cnt):
    top, w = cnt.most_common(1)[0]
    if top == 'intact':
        return 'intact'
    return top if w >= RATIOS[top] * sum(cnt.values()) else 'intact'

def coletar(pt, video_path, imgsz=SIZE, conf=CONF):
    """Passada 1: junta os votos de cada grão no vídeo TODO + guarda dets por frame."""
    model = YOLO(pt)
    votes = defaultdict(Counter)
    seen = Counter()
    dets_per_frame = defaultdict(list)
    k = 0
    for r in model.track(source=video_path, imgsz=imgsz, conf=conf, iou=0.5,
                         agnostic_nms=True, tracker='bytetrack.yaml',
                         persist=True, stream=True, verbose=False):
        if r.boxes.id is not None:
            for xyxy, tid, c, cf in zip(r.boxes.xyxy.cpu().numpy().astype(int),
                                        r.boxes.id.int().tolist(),
                                        r.boxes.cls.int().tolist(),
                                        r.boxes.conf.tolist()):
                votes[tid][NAMES[c]] += cf
                seen[tid] += 1
                x1, y1, x2, y2 = xyxy
                dets_per_frame[k].append((tid, x1, y1, x2, y2))
        k += 1
    # veredito final por grão — sobre o vídeo todo (sem hold, sem "analisando")
    verdict = {tid: veredito(v) for tid, v in votes.items()
               if seen[tid] >= MIN_TRACK_FRAMES}
    return dets_per_frame, verdict, seen, k

def render(video_path, dets_per_frame, verdict, out_path):
    """Passada 2: desenha cada grão já com o veredito final, caixa suavizada."""
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS) or 30
    smooth = {}
    writer, k = None, 0
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        if writer is None:
            h, w = frame.shape[:2]
            writer = cv2.VideoWriter(out_path, cv2.VideoWriter_fourcc(*'mp4v'), fps, (w, h))
        for tid, x1, y1, x2, y2 in dets_per_frame.get(k, []):
            if tid not in verdict:      # track curto demais (ruído) -> não desenha
                continue
            if tid in smooth:           # suaviza a caixa (não treme frame a frame)
                px1, py1, px2, py2 = smooth[tid]
                x1 = int(SMOOTH * x1 + (1 - SMOOTH) * px1)
                y1 = int(SMOOTH * y1 + (1 - SMOOTH) * py1)
                x2 = int(SMOOTH * x2 + (1 - SMOOTH) * px2)
                y2 = int(SMOOTH * y2 + (1 - SMOOTH) * py2)
            smooth[tid] = (x1, y1, x2, y2)
            cls = verdict[tid]
            color = COLORS[cls]
            cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
            cv2.putText(frame, f'#{tid} {PT_LABEL[cls]}', (x1, max(18, y1 - 6)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)
        writer.write(frame)
        k += 1
    cap.release(); writer.release()

MODELOS_TESTE = {'campeao_v2': CHAMPION_PT, 'novo_v3': DST}
resultados = {}
for tag, pt in MODELOS_TESTE.items():
    print(f'>>> {tag}: passada 1 (votos) em {os.path.basename(VIDEO_TESTE)}…')
    dets, verdict, seen, n_frames = coletar(pt, VIDEO_TESTE)
    print('    passada 2 (render)…')
    out_path = f'/content/drive/MyDrive/comparativo_{tag}.mp4'
    render(VIDEO_TESTE, dets, verdict, out_path)
    dist = Counter(verdict.values())
    resultados[tag] = dist
    print(f'    {n_frames} frames | {sum(dist.values())} grãos c/ veredito | {dict(dist)}')
    print(f'    salvo: {out_path}')

print()
print('=== resumo lado a lado ===')
for cls in NAMES:
    a = resultados['campeao_v2'].get(cls, 0)
    b = resultados['novo_v3'].get(cls, 0)
    print(f'  {PT_LABEL[cls]:14s} campeão_v2={a:3d}  novo_v3={b:3d}')
print()
print('Baixe comparativo_campeao_v2.mp4 e comparativo_novo_v3.mp4 do Drive e')
print('assista lado a lado: menos caixa piscando/classe trocando = melhor.')


## Depois do treino

1. **Teste no vídeo real** antes de adotar: rode o `vigil_deck.py` com o v3 e compare
   com o v2 no mesmo cenário (o placar mAP é no domínio das capturas; o juiz final é o vídeo).
2. Se o v3 alucinar menos/acertar mais → substitui o `.pt` no Deck/Windows e o OpenVINO.
3. O ciclo continua: mais capturas revisadas → v4. Cada rodada dessas é o
   aprendizado ativo do projeto rodando de ponta a ponta.